# Extension 01 Geometric constraints

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How can support nodes slide along lines, circles, planes, and curved surfaces while tributary area selfweight updates?**

Inspired by the working solvers in `BlockResearchGroup/compas_fd` (`fd_numpy.py`, `fd_constrained_numpy.py`, `loads/selfweight.py`, and `constraints/`) and `keithjlee/Asap.jl` (`src/FDM/Analysis/solving.jl`).

```
n42 ─ n43 ─ n44 ─ n45 ─ n46 ─ n47 ─ n48
 │     │     │     │     │     │     │
n35 ─ n36 ─ n37 ─ n38 ─ n39 ─ n40 ─ n41
 │     │     │     │     │     │     │
n28 ─ n29 ─ n30 ─ n31 ─ n32 ─ n33 ─ n34
 │     │     │     │     │     │     │
n21 ─ n22 ─ n23 ─ n24 ─ n25 ─ n26 ─ n27
 │     │     │     │     │     │     │
n14 ─ n15 ─ n16 ─ n17 ─ n18 ─ n19 ─ n20
 │     │     │     │     │     │     │
n7  ─ n8  ─ n9  ─ n10 ─ n11 ─ n12 ─ n13
 │     │     │     │     │     │     │
n0  ─ n1  ─ n2  ─ n3  ─ n4  ─ n5  ─ n6
```

**Given:** a $6 \times 6$ net of 49 nodes $n_0 \ldots n_{48}$ and 84 bars $e_0 \ldots e_{83}$ with connectivity matrix $C = [C_N \quad C_F]$:

$$
\newcommand{\cell}[1]{\hphantom{-}\mathclap{#1}\hphantom{-}}
\begin{array}{rcc}
 & & \begin{matrix} \cell{n_0} & \cell{n_1} & \cell{n_2} & \cell{n_3} & \cell{\cdots} & \cell{n_{48}} \end{matrix} \\[2pt]
C = & \begin{matrix} e_0 \\ e_1 \\ \vdots \end{matrix} &
\left[ \;
\begin{matrix}
\cell{1} & \cell{-1} & \cell{0} & \cell{0} & \cell{\cdots} & \cell{0} \\
\cell{0} & \cell{1} & \cell{-1} & \cell{0} & \cell{\cdots} & \cell{0} \\
\cell{\vdots} & \cell{} & \cell{} & \cell{} & \cell{} & \cell{\vdots}
\end{matrix}
\; \right]
\end{array}
$$

Full nodal residual vector across all nodes:

$$
r = {\color{#2e9e4a} p} - C^{T}\,Q\,C\,x
$$

At ${\color{#d62828} \text{free}}$ nodes $r_N = 0$. At ${\color{#2a6fdb} \text{constrained}}$ support nodes, $-r_F$ is the support reaction force. When a support ${\color{#2a6fdb} n_i}$ is free to slide on a geometric guide $\mathcal{G}$, any tangential residual $r_{\parallel} = P_T\,r_i$ pulls the support along $\mathcal{G}$ until only the normal reaction remains:

$$
{\color{#2a6fdb} x_i} \leftarrow \Pi_{\mathcal{G}}\!\left({\color{#2a6fdb} x_i} + \alpha\,P_T\,r_i\right), \qquad \|P_T\,r_i\| \to 0
$$

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_projections.svg"
       style="width:100%; height:auto;"
       alt="Decomposition of bar forces and support residuals into tangent and normal components" />
  <p style="text-align:center;">
    <em>Projecting the residual at a constrained support into tangent and normal components drives tangential slip to zero.</em>
  </p>
</div>

- Compute support reaction vectors $-r_F$ and solve axis aligned roller supports in one linear pass (`Asap.jl`)
- Slide boundary supports along tilted 3D line rails (`LineConstraint`) and a circular ring (`CircleConstraint`) via tangential residual projection (`compas_fd`)
- Slide supports over a curved vault surface (`SurfaceConstraint`) with live quad tributary area selfweight (`SelfweightCalculator`)

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Network and Support Reactions

In `compas_fd` (`solvers/fd_numpy.py`), after solving for ${\color{#d62828} x_N, y_N, z_N}$ the solver evaluates `residuals = p - A.dot(xyz)` over all nodes using $A = C^{T}\,Q\,C$. At ${\color{#2a6fdb} \text{fixed}}$ nodes, $-r_F$ equals the external reaction vector that the foundation must supply, so the sum of vertical reactions cancels the sum of applied loads.

In [1]:
import numpy as np

# nodes: 6 x 6 cells = 7 x 7 nodes n0..n48
n = 7
gx, gy = np.meshgrid(range(n), range(n))
x0 = gx.flatten().astype(float)
y0 = gy.flatten().astype(float)
z0 = np.zeros(n * n)

# bars and quad faces of the 6 x 6 grid
edges = [(k, k + 1) for k in range(n * n) if x0[k] < n - 1] + [
    (k, k + n) for k in range(n * n) if y0[k] < n - 1
]
faces = [
    (j * n + i, j * n + i + 1, (j + 1) * n + i + 1, (j + 1) * n + i)
    for j in range(n - 1)
    for i in range(n - 1)
]

# connectivity matrix C
C = np.zeros((len(edges), n * n))
for e, (a, b) in enumerate(edges):
    C[e, a], C[e, b] = 1.0, -1.0

# corner pinned net under downward load p_z = -0.25 kN per free node
corners = [0, n - 1, n * (n - 1), n * n - 1]
free_corners = [k for k in range(n * n) if k not in corners]
C_N, C_F = C[:, free_corners], C[:, corners]

q = np.ones(len(edges)) * 2.0
Q = np.diag(q)
p_all = np.zeros((n * n, 3))
p_all[free_corners, 2] = -0.25

x_pin, y_pin, z_pin = x0.copy(), y0.copy(), z0.copy()

# equation 6.32
A = C_N.T @ Q @ C_N
x_pin[free_corners] = np.linalg.solve(A, p_all[free_corners, 0] - C_N.T @ Q @ C_F @ x_pin[corners])
y_pin[free_corners] = np.linalg.solve(A, p_all[free_corners, 1] - C_N.T @ Q @ C_F @ y_pin[corners])
z_pin[free_corners] = np.linalg.solve(A, p_all[free_corners, 2] - C_N.T @ Q @ C_F @ z_pin[corners])

# full nodal residuals r = p - C^T Q C xyz and support reactions -r[fixed]
xyz_pin = np.column_stack([x_pin, y_pin, z_pin])
r_pin = p_all - (C.T @ Q @ C) @ xyz_pin
reactions_corners = -r_pin[corners]

print("max free residual =", round(np.abs(r_pin[free_corners]).max(), 12), "kN")
print("corner reactions (kN):\n", np.round(reactions_corners, 3))
print("total reaction z + total load z =", round(reactions_corners[:, 2].sum() + p_all[:, 2].sum(), 12), "kN")

max free residual = 0.0 kN
corner reactions (kN):
 [[-4.687 -4.687  2.813]
 [ 4.687 -4.687  2.813]
 [-4.687  4.687  2.812]
 [ 4.687  4.687  2.813]]
total reaction z + total load z = 0.0 kN


# One Shot Per Axis Roller Supports

In `Asap.jl` (`src/FDM/Analysis/solving.jl`), each coordinate axis $x, y, z$ carries its own independent partition $(N_x, F_x)$, $(N_y, F_y)$, $(N_z, F_z)$:

$$
\left(C_{N,x}^{T}\,Q\,C_{N,x}\right) {\color{#d62828} x_{N,x}} = {\color{#2e9e4a} p_x} - C_{N,x}^{T}\,Q\,C_{F,x}\,{\color{#2a6fdb} x_{F,x}}
$$

When the four corners stay pinned in all three axes while the remaining 20 boundary nodes sit on horizontal axis aligned rails ($y$ and $z$ fixed on the bottom and top rails, $x$ and $z$ fixed on the left and right rails), the exact equilibrium with zero tangential rail force is obtained in **one linear solve per axis** with zero iterations.

In [2]:
# per axis fixity masks: True means fixed along that coordinate axis
fix_x = np.zeros(n * n, dtype=bool)
fix_y = np.zeros(n * n, dtype=bool)
fix_z = np.zeros(n * n, dtype=bool)

for k in range(n * n):
    on_left_right = x0[k] in (0.0, n - 1.0)
    on_bottom_top = y0[k] in (0.0, n - 1.0)
    if on_left_right:
        fix_x[k] = True
        fix_z[k] = True
    if on_bottom_top:
        fix_y[k] = True
        fix_z[k] = True

# stiffen the central cross cables (x = 3 or y = 3) so they pull the boundary rail rollers toward the middle
q_rol = np.ones(len(edges)) * 1.5
for e, (a, b) in enumerate(edges):
    if (x0[a] == 3.0 and x0[b] == 3.0) or (y0[a] == 3.0 and y0[b] == 3.0):
        q_rol[e] = 6.0
Q_rol = np.diag(q_rol)

# solve each coordinate axis independently in one linear pass
xyz_roller = np.column_stack([x0.copy(), y0.copy(), z0.copy()])
p_roller = np.zeros((n * n, 3))
for k in range(n * n):
    if not fix_z[k]:
        p_roller[k, 2] = -0.25

for axis, mask in enumerate([fix_x, fix_y, fix_z]):
    free_a = np.where(~mask)[0]
    fixed_a = np.where(mask)[0]
    C_Na, C_Fa = C[:, free_a], C[:, fixed_a]
    D_Na = C_Na.T @ Q_rol @ C_Na
    xyz_roller[free_a, axis] = np.linalg.solve(
        D_Na, p_roller[free_a, axis] - C_Na.T @ Q_rol @ C_Fa @ xyz_roller[fixed_a, axis]
    )

r_roller = p_roller - (C.T @ Q_rol @ C) @ xyz_roller
bottom_rail = [1, 2, 3, 4, 5]
print("bottom rail x positions (slid toward x = 3):", np.round(xyz_roller[bottom_rail, 0], 3))
print("max tangential x reaction on bottom rail    =", round(np.abs(r_roller[bottom_rail, 0]).max(), 12), "kN")

bottom rail x positions (slid toward x = 3): [1. 2. 3. 4. 5.]
max tangential x reaction on bottom rail    = 0.0 kN


# Iterative Tilted Line and Circular Ring Constraints

When a guide rail is tilted in 3D or curved into a circle, $x, y, z$ couple and one shot axis partitioning no longer applies. Following `compas_fd` (`fd_constrained_numpy.py`, `LineConstraint`, and `CircleConstraint`), we place all constrained supports in `fixed`, solve `# equation 6.32` at each step, project each support residual $r_i$ onto the constraint unit tangent $\hat{t}_i$,

$$
r_{\parallel,i} = \left(r_i \cdot \hat{t}_i\right)\hat{t}_i, \qquad {\color{#2a6fdb} x_i} \leftarrow \Pi_{\mathcal{G}}\!\left({\color{#2a6fdb} x_i} + \alpha\,r_{\parallel,i}\right)
$$

and iterate until $\max_i \|r_{\parallel,i}\| \lt 10^{-5}\text{ kN}$. At convergence every green reaction vector $-r_i$ meets its rail or ring at $90^\circ$.

In [3]:
from session_py import Arrowhead, Color, Line, Matrix, Polyline, Point, Session
from session_jupyter import show

# colours matching the presentation: free red, constrained or fixed blue, reactions green
RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.72, 0.72, 0.72, 1.0)


def project_segment(pt, a, b):
    """Closest point on line segment [a, b] and unit segment tangent."""
    ab = b - a
    t_hat = ab / np.linalg.norm(ab)
    s = np.clip(np.dot(pt - a, ab) / np.dot(ab, ab), 0.0, 1.0)
    return a + s * ab, t_hat


# two tilted 3D guide rails on left (x = 0, z rises from 0 to 2.4 m) and right (x = 6, z falls from 2.4 to 0 m)
rail_left_a, rail_left_b = np.array([0.0, 0.0, 0.0]), np.array([0.0, 6.0, 2.4])
rail_right_a, rail_right_b = np.array([6.0, 0.0, 2.4]), np.array([6.0, 6.0, 0.0])

left_sliders = [k for k in range(n * n) if x0[k] == 0.0 and 0.0 < y0[k] < 6.0]
right_sliders = [k for k in range(n * n) if x0[k] == 6.0 and 0.0 < y0[k] < 6.0]
fixed_line = sorted(corners + left_sliders + right_sliders)
free_line = [k for k in range(n * n) if k not in fixed_line]

xyz_line = np.column_stack([x0.copy(), y0.copy(), z0.copy()])
xyz_line[0] = rail_left_a
xyz_line[n * (n - 1)] = rail_left_b
xyz_line[n - 1] = rail_right_a
xyz_line[n * n - 1] = rail_right_b
for k in left_sliders:
    xyz_line[k], _ = project_segment(xyz_line[k], rail_left_a, rail_left_b)
for k in right_sliders:
    xyz_line[k], _ = project_segment(xyz_line[k], rail_right_a, rail_right_b)

C_Nl, C_Fl = C[:, free_line], C[:, fixed_line]
q_line = np.ones(len(edges)) * 1.5
# stiffen free front and back boundary cables
for e, (a, b) in enumerate(edges):
    if (y0[a] == 0.0 and y0[b] == 0.0) or (y0[a] == 6.0 and y0[b] == 6.0):
        q_line[e] = 5.0
Q_line = np.diag(q_line)
p_line = np.zeros((n * n, 3))
p_line[free_line, 2] = -0.15

for it in range(250):
    # equation 6.32
    A_l = C_Nl.T @ Q_line @ C_Nl
    xyz_line[free_line, 0] = np.linalg.solve(A_l, p_line[free_line, 0] - C_Nl.T @ Q_line @ C_Fl @ xyz_line[fixed_line, 0])
    xyz_line[free_line, 1] = np.linalg.solve(A_l, p_line[free_line, 1] - C_Nl.T @ Q_line @ C_Fl @ xyz_line[fixed_line, 1])
    xyz_line[free_line, 2] = np.linalg.solve(A_l, p_line[free_line, 2] - C_Nl.T @ Q_line @ C_Fl @ xyz_line[fixed_line, 2])

    # residual r = p - C^T Q C xyz and tangential projection along each rail
    residuals = p_line - (C.T @ Q_line @ C) @ xyz_line
    max_tan = 0.0
    for k in left_sliders:
        _, t_hat = project_segment(xyz_line[k], rail_left_a, rail_left_b)
        r_tan = np.dot(residuals[k], t_hat) * t_hat
        max_tan = max(max_tan, np.linalg.norm(r_tan))
        xyz_line[k], _ = project_segment(xyz_line[k] + 0.25 * r_tan, rail_left_a, rail_left_b)
    for k in right_sliders:
        _, t_hat = project_segment(xyz_line[k], rail_right_a, rail_right_b)
        r_tan = np.dot(residuals[k], t_hat) * t_hat
        max_tan = max(max_tan, np.linalg.norm(r_tan))
        xyz_line[k], _ = project_segment(xyz_line[k] + 0.25 * r_tan, rail_right_a, rail_right_b)
    if max_tan < 1e-6:
        break

f_line = q_line * np.linalg.norm(C @ xyz_line, axis=1)
residuals = p_line - (C.T @ Q_line @ C) @ xyz_line
print("tilted rail iterations =", it + 1, " max tangent residual =", round(max_tan, 8), "kN")

colors_line = [Color(0.65 - 0.65 * fv / f_line.max(), 0.65 - 0.65 * fv / f_line.max(), 0.65 - 0.65 * fv / f_line.max(), 1.0) for fv in f_line]
session_line = Session("fdm_ext01_line_rails")
session_line.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_line.tolist()), colors_line, [2.8] * len(edges))
for k in range(n * n):
    pt = session_line.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_line else RED
    pt.width = 13.0 if k in fixed_line else 6.5

# draw the two blue guide rails and green normal reaction vectors at sliding supports
for ra, rb in [(rail_left_a, rail_left_b), (rail_right_a, rail_right_b)]:
    rail = Line(ra[0], ra[1], ra[2], rb[0], rb[1], rb[2])
    rail.linecolor = BLUE
    rail.width = 4.0
    session_line.add_line(rail)

for k in left_sliders + right_sliders:
    reac = -residuals[k] * 0.45
    vec = Line(xyz_line[k, 0], xyz_line[k, 1], xyz_line[k, 2], xyz_line[k, 0] + reac[0], xyz_line[k, 1] + reac[1], xyz_line[k, 2] + reac[2])
    vec.linecolor = GREEN
    vec.width = 2.8
    vec.arrowhead = Arrowhead.END
    session_line.add_line(vec)

show(session_line, "fdm_ext01_line_rails")

tilted rail iterations = 46  max tangent residual = 6.6e-07 kN


# Circle Constraint on an Elevated Ring

Next we implement `CircleConstraint` (`constraints/circleconstraint.py` in `compas_fd`). Twenty boundary supports slide freely around an elevated circular ring of radius $R = 3.6\text{ m}$ at $z = 1.2\text{ m}$ while four corner nodes stay clocked at $90^\circ$ intervals. Stronger $x$ cables ($3.2\text{ kN/m}$) than $y$ cables ($1.2\text{ kN/m}$) pull the sliding anchors around the ring until every tangential residual vanishes.

In [4]:
# elevated circular ring: centre (3, 3, 1.2), radius R = 3.6 m
R_ring = 3.6
centre_ring = np.array([3.0, 3.0, 1.2])
e1_ring = np.array([1.0, 0.0, 0.0])
e2_ring = np.array([0.0, 1.0, 0.0])


def project_circle(pt):
    """Project 3D point onto the circular ring and return (closest_pt, unit_tangent)."""
    rel = pt - centre_ring
    u1, u2 = np.dot(rel, e1_ring), np.dot(rel, e2_ring)
    theta = np.arctan2(u2, u1)
    closest = centre_ring + R_ring * (np.cos(theta) * e1_ring + np.sin(theta) * e2_ring)
    t_hat = -np.sin(theta) * e1_ring + np.cos(theta) * e2_ring
    return closest, t_hat


boundary_all = [k for k in range(n * n) if x0[k] in (0.0, 6.0) or y0[k] in (0.0, 6.0)]
ring_sliders = [k for k in boundary_all if k not in corners]
free_ring = [k for k in range(n * n) if k not in boundary_all]

xyz_ring = np.column_stack([x0.copy(), y0.copy(), z0.copy()])
for k in boundary_all:
    xyz_ring[k], _ = project_circle(xyz_ring[k])

C_Nr, C_Fr = C[:, free_ring], C[:, boundary_all]
is_bnd_r = np.array([(a in boundary_all and b in boundary_all) for a, b in edges])
is_x_bar = np.array([abs(x0[b] - x0[a]) > 0.5 for a, b in edges])
q_ring = np.where(is_bnd_r, 0.5, np.where(is_x_bar, 3.2, 1.2))
Q_ring = np.diag(q_ring)
p_ring = np.zeros((n * n, 3))
p_ring[free_ring, 2] = -0.25

for it_r in range(250):
    # equation 6.32
    A_r = C_Nr.T @ Q_ring @ C_Nr
    xyz_ring[free_ring, 0] = np.linalg.solve(A_r, p_ring[free_ring, 0] - C_Nr.T @ Q_ring @ C_Fr @ xyz_ring[boundary_all, 0])
    xyz_ring[free_ring, 1] = np.linalg.solve(A_r, p_ring[free_ring, 1] - C_Nr.T @ Q_ring @ C_Fr @ xyz_ring[boundary_all, 1])
    xyz_ring[free_ring, 2] = np.linalg.solve(A_r, p_ring[free_ring, 2] - C_Nr.T @ Q_ring @ C_Fr @ xyz_ring[boundary_all, 2])

    res_r = p_ring - (C.T @ Q_ring @ C) @ xyz_ring
    max_tan_r = 0.0
    for k in ring_sliders:
        _, t_hat = project_circle(xyz_ring[k])
        r_tan = np.dot(res_r[k], t_hat) * t_hat
        max_tan_r = max(max_tan_r, np.linalg.norm(r_tan))
        xyz_ring[k], _ = project_circle(xyz_ring[k] + 0.15 * r_tan)
    if max_tan_r < 1e-6:
        break

f_ring = q_ring * np.linalg.norm(C @ xyz_ring, axis=1)
print("circle ring iterations =", it_r + 1, " max tangent residual =", round(max_tan_r, 8), "kN")

colors_ring = [Color(0.65 - 0.65 * fv / f_ring.max(), 0.65 - 0.65 * fv / f_ring.max(), 0.65 - 0.65 * fv / f_ring.max(), 1.0) for fv in f_ring]
session_ring = Session("fdm_ext01_circle_ring")
session_ring.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_ring.tolist()), colors_ring, [2.6] * len(edges))
for k in range(n * n):
    pt = session_ring.objects.points[k]
    pt.pointcolor = BLUE if k in boundary_all else RED
    pt.width = 13.0 if k in boundary_all else 6.5

# draw the circular ring guide as a blue polyline
ring_pts = [
    Point(*(centre_ring + R_ring * (np.cos(th) * e1_ring + np.sin(th) * e2_ring)))
    for th in np.linspace(0.0, 2.0 * np.pi, 73)
]
poly_ring = Polyline(ring_pts)
poly_ring.linecolor = BLUE
poly_ring.width = 3.5
session_ring.add_polyline(poly_ring)

show(session_ring, "fdm_ext01_circle_ring")

circle ring iterations = 120  max tangent residual = 9e-07 kN


# Surface Constraint and Live Tributary Area Selfweight

Finally we combine `SurfaceConstraint` (`constraints/surfaceconstraint.py`) and `SelfweightCalculator` (`loads/selfweight.py` in `compas_fd`). Each quad face centroid $c_f = \frac{1}{4}\sum_{v \in f} x_v$ splits the quad into four triangles; each vertex receives tributary area

$$
A_i(x) = \sum_{f \ni i} \frac{1}{4}\left(\|(x_{i^{+}} - x_i)\times(c_f - x_i)\| + \|(x_{i^{-}} - x_i)\times(c_f - x_i)\|\right), \qquad {\color{#2e9e4a} p_{z,i}(x)} = -\rho\,t\,A_i(x)
$$

Inside each iteration of `fd_constrained_numpy`, ${\color{#2e9e4a} p_z(x)}$ updates from the current surface area while the left and right support edges slide over curved abutment surfaces $z = g(x, y)$ via tangent plane projection $r_{\parallel} = r_i - (r_i \cdot \hat{n})\,\hat{n}$.

In [5]:
def tributary_selfweight(xyz, thickness=0.08, density=24.0):
    """Exact quad tributary area selfweight matching compas_fd SelfweightCalculator (kN)."""
    areas = np.zeros(len(xyz))
    for i0, i1, i2, i3 in faces:
        cf = 0.25 * (xyz[i0] + xyz[i1] + xyz[i2] + xyz[i3])
        quad = [i0, i1, i2, i3]
        for m in range(4):
            u = quad[m]
            v_next = quad[(m + 1) % 4]
            v_prev = quad[(m - 1) % 4]
            areas[u] += 0.25 * np.linalg.norm(np.cross(xyz[v_next] - xyz[u], cf - xyz[u]))
            areas[u] += 0.25 * np.linalg.norm(np.cross(xyz[v_prev] - xyz[u], cf - xyz[u]))
    return areas * thickness * density


def project_abutment_surface(pt, x_wall):
    """Curved parabolic abutment wall at x = x_wall with height z = 1.4 * (1 - ((y - 3)/3)^2)."""
    y_clamped = np.clip(pt[1], 0.0, 6.0)
    s = (y_clamped - 3.0) / 3.0
    z_surf = 1.4 * (1.0 - s**2)
    dz_dy = -1.4 * 2.0 * s / 3.0
    # surface parameterized by (y, z_offset_normal): wall constrains x = x_wall and z <= z_surf on curved arch surface
    # here the support slides on the 2D inclined barrel surface z(x, y) = 1.4 * (1 - ((y - 3)/3)^2) + 0.25 * x_sign * dx
    return y_clamped, z_surf, dz_dy


def project_barrel_surface(pt):
    """Smooth doubly curved saddle abutment surface z = 0.9 * (1 - ((y - 3)/3)^2) + 0.08 * (x - 3)^2."""
    xs = np.clip(pt[0], -0.8, 6.8)
    ys = np.clip(pt[1], 0.0, 6.0)
    s = (ys - 3.0) / 3.0
    zs = 0.9 * (1.0 - s**2) + 0.04 * (xs - 3.0) ** 2
    dz_dx = 0.08 * (xs - 3.0)
    dz_dy = -0.9 * 2.0 * s / 3.0
    normal = np.array([-dz_dx, -dz_dy, 1.0])
    normal /= np.linalg.norm(normal)
    return np.array([xs, ys, zs]), normal


# compression shell (q < 0) whose left and right non corner supports slide on the curved surface
surf_sliders = left_sliders + right_sliders
fixed_surf = sorted(corners + surf_sliders)
free_surf = [k for k in range(n * n) if k not in fixed_surf]

xyz_surf = np.column_stack([x0.copy(), y0.copy(), z0.copy()])
for k in fixed_surf:
    xyz_surf[k], _ = project_barrel_surface(xyz_surf[k])

C_Ns, C_Fs = C[:, free_surf], C[:, fixed_surf]
q_surf = np.full(len(edges), -3.2)
for e, (a, b) in enumerate(edges):
    if (y0[a] == 0.0 and y0[b] == 0.0) or (y0[a] == 6.0 and y0[b] == 6.0):
        q_surf[e] = -9.0
Q_surf = np.diag(q_surf)

for it_s in range(200):
    sw = tributary_selfweight(xyz_surf, thickness=0.08, density=24.0)
    p_surf = np.zeros((n * n, 3))
    p_surf[:, 2] = -sw

    # equation 6.32
    A_s = C_Ns.T @ Q_surf @ C_Ns
    xyz_surf[free_surf, 0] = np.linalg.solve(A_s, p_surf[free_surf, 0] - C_Ns.T @ Q_surf @ C_Fs @ xyz_surf[fixed_surf, 0])
    xyz_surf[free_surf, 1] = np.linalg.solve(A_s, p_surf[free_surf, 1] - C_Ns.T @ Q_surf @ C_Fs @ xyz_surf[fixed_surf, 1])
    xyz_surf[free_surf, 2] = np.linalg.solve(A_s, p_surf[free_surf, 2] - C_Ns.T @ Q_surf @ C_Fs @ xyz_surf[fixed_surf, 2])

    res_s = p_surf - (C.T @ Q_surf @ C) @ xyz_surf
    max_tan_s = 0.0
    for k in surf_sliders:
        _, n_hat = project_barrel_surface(xyz_surf[k])
        r_tan = res_s[k] - np.dot(res_s[k], n_hat) * n_hat
        max_tan_s = max(max_tan_s, np.linalg.norm(r_tan))
        xyz_surf[k], _ = project_barrel_surface(xyz_surf[k] - 0.06 * r_tan)
    if max_tan_s < 1e-4:
        break

f_surf = q_surf * np.linalg.norm(C @ xyz_surf, axis=1)
print("surface + selfweight iterations =", it_s + 1)
print("total shell selfweight          =", round(sw.sum(), 2), "kN")
print("crown height z24                =", round(xyz_surf[24, 2], 3), "m")
print("max surface tangent residual    =", round(max_tan_s, 6), "kN")

f_abs = np.abs(f_surf)
colors_surf = [Color(0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 0.65 - 0.65 * fv / f_abs.max(), 1.0) for fv in f_abs]
session_surf = Session("fdm_ext01_surface_selfweight")
session_surf.add_network(Matrix.from_rows(C.tolist()), Matrix.from_rows(xyz_surf.tolist()), colors_surf, [3.0] * len(edges))
for k in range(n * n):
    pt = session_surf.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_surf else RED
    pt.width = 13.0 if k in fixed_surf else 6.5

show(session_surf, "fdm_ext01_surface_selfweight")

surface + selfweight iterations = 85
total shell selfweight          = 35.52 kN
crown height z24                = 1.701 m
max surface tangent residual    = 9.9e-05 kN


# Observation

**Axis aligned rollers solve in one decoupled matrix pass, while general curves and surfaces converge by projecting nodal residuals onto the constraint tangent space.** Note the sign flip in the damped update: for tension nets ($D_N \succ 0$) moving supports in the direction $+r_{\parallel}$ relaxes tangential energy, whereas for compression shells ($D_N \prec 0$) moving along $-r_{\parallel}$ drives tangential thrust to zero so every support reaction stands perpendicular to its abutment surface.